# Лабораторная работа: HITL сегментация

Запустите ячейки сверху вниз. Полученные метрики внесите в вывод.

In [ ]:
import numpy as np, pandas as pd, matplotlib.pyplot as plt, tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
keras.utils.set_random_seed(42)
print('GPU:',tf.config.list_physical_devices('GPU'))

In [ ]:
from google.colab import files
from PIL import Image
from pathlib import Path
import io,zipfile,tempfile,os
u=files.upload();name=next(n for n in u if n.endswith('.zip'))
root=tempfile.mkdtemp()
with zipfile.ZipFile(io.BytesIO(u[name])) as z:
 for member in z.infolist():
  dest=os.path.realpath(os.path.join(root,member.filename))
  assert dest.startswith(os.path.realpath(root)+os.sep) or dest==os.path.realpath(root)
 z.extractall(root)
imgs={p.stem:p for p in Path(root).rglob('*') if 'images' in p.parts and p.suffix.lower() in ('.png','.jpg','.jpeg')}
masks={p.stem:p for p in Path(root).rglob('*') if 'masks' in p.parts and p.suffix.lower() in ('.png','.jpg','.jpeg')}
keys=sorted(imgs.keys()&masks.keys());assert len(keys)>3,'Нужен ZIP с images/ и masks/'
X=np.array([np.asarray(Image.open(imgs[k]).convert('RGB').resize((128,128)))/255 for k in keys],dtype='float32')
raw=np.array([np.asarray(Image.open(masks[k]).convert('RGB').resize((128,128),Image.Resampling.NEAREST)) for k in keys],dtype='uint8')
palette,inv=np.unique(raw.reshape(-1,3),axis=0,return_inverse=True);Y=inv.reshape(raw.shape[:3]).astype('int32')
from sklearn.model_selection import train_test_split
a,b,c,d=train_test_split(X,Y,test_size=.2,random_state=42)
def conv(x,n):
 x=layers.Conv2D(n,3,activation='relu',padding='same')(x)
 return layers.Conv2D(n,3,activation='relu',padding='same')(x)
inp=keras.Input((128,128,3));s1=conv(inp,32);s2=conv(layers.MaxPooling2D()(s1),64)
x=conv(layers.MaxPooling2D()(s2),128)
x=conv(layers.Concatenate()([layers.UpSampling2D()(x),s2]),64)
x=conv(layers.Concatenate()([layers.UpSampling2D()(x),s1]),32)
model=keras.Model(inp,layers.Conv2D(len(palette),1,activation='softmax')(x))
model.compile('adam','sparse_categorical_crossentropy',metrics=['accuracy'])
history=model.fit(a,c,validation_data=(b,d),epochs=20,batch_size=8)
pd.DataFrame(history.history).plot(grid=True);plt.show()